<a href="https://colab.research.google.com/github/springboard5678x/Emotion-Detection-and-Music-Recommended-system_Batch-27_nov/blob/Ranit-Deria/Notebooks/MoodMate_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')


base_path = '/content/drive/MyDrive/MoodMate/datasets'
fer_path = os.path.join(base_path, 'fer2013.csv')
music_path = os.path.join(base_path, 'dataset.csv')

print(f"Checking files in {base_path}...")
if os.path.exists(fer_path) and os.path.exists(music_path):
    print("✅ Raw files found! Starting processing...")
else:
    print(f"❌ ERROR: Files not found. Please check your Drive folder contains 'fer2013.csv' and 'dataset.csv'.")





print("\n--- Processing FER-2013 Images ---")
try:
    data = pd.read_csv(fer_path)
    pixels = data['pixels'].tolist()
    width, height = 48, 48

    faces = []
    for pixel_sequence in pixels:
        face = [int(pixel) for pixel in pixel_sequence.split(' ')]
        face = np.asarray(face).reshape(width, height)
        faces.append(face.astype('float32') / 255.0)


    faces = np.asarray(faces)
    faces = np.expand_dims(faces, -1)


    emotions = pd.get_dummies(data['emotion']).values



    np.save(os.path.join(base_path, 'X_faces.npy'), faces)
    np.save(os.path.join(base_path, 'y_emotions.npy'), emotions)
    print(f"✅ Images processed: {faces.shape}")
    print(f"✅ Saved: X_faces.npy, y_emotions.npy")

except Exception as e:
    print(f"❌ Error processing images: {e}")





print("\n--- Processing Spotify Music Data ---")
try:
    music_df = pd.read_csv(music_path)


    def get_mood_from_audio(row):
        v = row['valence']
        e = row['energy']

        if v >= 0.5 and e >= 0.5: return 3   # Happy
        elif v < 0.5 and e < 0.5: return 4   # Sad
        elif v < 0.5 and e >= 0.5: return 0  # Angry
        elif v >= 0.5 and e < 0.5: return 6  # Calm/Neutral
        return -1


    if 'valence' in music_df.columns and 'energy' in music_df.columns:
        music_df['emotion_id'] = music_df.apply(get_mood_from_audio, axis=1)


        cols_to_keep = ['track_name', 'artists', 'album_name', 'track_genre', 'valence', 'energy', 'emotion_id']

        final_cols = [c for c in cols_to_keep if c in music_df.columns]

        clean_music = music_df[final_cols]
        clean_music = clean_music[clean_music['emotion_id'] != -1]

        clean_music.to_csv(os.path.join(base_path, 'processed_music.csv'), index=False)
        print(f"✅ Music processed: {len(clean_music)} songs mapped.")
        print(f"✅ Saved: processed_music.csv")
    else:
        print("❌ Error: 'valence' or 'energy' columns missing in dataset.csv")

except Exception as e:
    print(f"❌ Error processing music: {e}")

print("\n🎉 PHASE 2 COMPLETE")

Mounted at /content/drive
Checking files in /content/drive/MyDrive/MoodMate/datasets...
✅ Raw files found! Starting processing...

--- Processing FER-2013 Images ---
✅ Images processed: (35887, 48, 48, 1)
✅ Saved: X_faces.npy, y_emotions.npy

--- Processing Spotify Music Data ---
✅ Music processed: 114000 songs mapped.
✅ Saved: processed_music.csv

🎉 PHASE 2 COMPLETE


In [ ]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Assuming `faces` and `emotions` are already loaded from the previous step
# Split the data into training and testing sets
# Using a dummy X_data for splitting, as the focus is on y_train/y_test
# You would typically split `faces` as X_train, X_test here as well.

# Placeholder for X_data for train_test_split. In a full model, you would split `faces`.
# For now, we only need to split 'emotions' to resolve the NameError for y_train/y_test
# We create a dummy X to match the dimensions for train_test_split
X_dummy = np.zeros(emotions.shape)

_, _, y_train, y_test = train_test_split(X_dummy, emotions, test_size=0.2, random_state=42, stratify=np.argmax(emotions, axis=1))

# Convert one-hot encoded labels to class indices
y_train_classes = np.argmax(y_train, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Get counts for training data
train_class_counts = Counter(y_train_classes)
print("Training data image counts per class:")
for i, label in enumerate(emotion_labels):
    print(f"  {label}: {train_class_counts[i]} images")

# Get counts for test data
test_class_counts = Counter(y_test_classes)
print("\nTest data image counts per class:")
for i, label in enumerate(emotion_labels):
    print(f"  {label}: {test_class_counts[i]} images")

Training data image counts per class:
  Angry: 3962 images
  Disgust: 438 images
  Fear: 4097 images
  Happy: 7191 images
  Sad: 4861 images
  Surprise: 3202 images
  Neutral: 4958 images

Test data image counts per class:
  Angry: 991 images
  Disgust: 109 images
  Fear: 1024 images
  Happy: 1798 images
  Sad: 1216 images
  Surprise: 800 images
  Neutral: 1240 images
